In [47]:
# STEP 1: Load the source Excel file
# Display the dataset size and first five rows

import pandas as pd

df = pd.read_excel("../data/task_2_data_ex.xlsx")

print(df.shape)
display(df.head())

(1320, 11)


,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.0,50000,8002.0,PROD,990.0,RLT_10
1,2024,1,50000,8002,PROD,859.0,80070,8007.0,PROD,879.0,RLT_10
2,2024,1,50000,8002,PROD,859.0,90000,NaN,ADD,50.0,RLT_10
3,2024,1,50000,8002,PROD,859.0,90001,NaN,ADD,20.0,RLT_10
4,2024,1,80070,8007,PROD,929.0,80010,8001.0,PROD,3626.0,RLT_10


In [48]:
# STEP 2: Explore the data
# Check column names, years, material types and missing values

print(df.columns.tolist())

print(df["year"].unique())

print(df["produced_material_release_type"].value_counts())

print(df.isna().sum())

['year', 'month', 'produced_material', 'produced_material_production_type', 'produced_material_release_type', 'produced_material_quantity', 'component_material', 'component_material_production_type', 'component_material_release_type', 'component_material_quantity', 'plant_id']
[2024]
produced_material_release_type
PROD    1200
FIN      120
Name: count, dtype: int64
year                                    0
month                                   0
produced_material                       0
produced_material_production_type       0
produced_material_release_type          0
produced_material_quantity              0
component_material                      0
component_material_production_type    840
component_material_release_type         0
component_material_quantity             0
plant_id                                0
dtype: int64


In [49]:
# STEP 3: Calculate annual production
# Remove repeated monthly production records, then sum by year, plant and material

def calculate_annual_production(df):

    columns = [
        "year",
        "month",
        "plant_id",
        "produced_material",
        "produced_material_release_type",
        "produced_material_production_type",
        "produced_material_quantity"
    ]

    production = df[columns].drop_duplicates()

    annual_production = production.groupby(
        [
            "year",
            "plant_id",
            "produced_material",
            "produced_material_release_type",
            "produced_material_production_type"
        ],
        dropna=False,
        as_index=False
    )["produced_material_quantity"].sum()

    return annual_production


annual_production = calculate_annual_production(df)

display(annual_production.head())

,year,plant_id,produced_material,produced_material_release_type,produced_material_production_type,produced_material_quantity
0,2024,RLT_10,10000,FIN,8002,11708.0
1,2024,RLT_10,10001,FIN,8002,12023.0
2,2024,RLT_10,50000,PROD,8002,9538.0
3,2024,RLT_10,50001,PROD,8002,9487.0
4,2024,RLT_10,80000,PROD,8000,23478.0


In [50]:
# STEP 4: Calculate annual component consumption
# Group each material-component relationship and sum its consumption for the year

def calculate_annual_components(df):

    columns = [
        "year",
        "plant_id",
        "produced_material",
        "produced_material_release_type",
        "produced_material_production_type",
        "component_material",
        "component_material_release_type",
        "component_material_production_type"
    ]

    annual_components = df.groupby(
        columns,
        dropna=False,
        as_index=False
    )["component_material_quantity"].sum()

    return annual_components


annual_components = calculate_annual_components(df)

display(annual_components.head())

,year,plant_id,produced_material,produced_material_release_type,produced_material_production_type,component_material,component_material_release_type,component_material_production_type,component_material_quantity
0,2024,RLT_10,10000,FIN,8002,50000,PROD,8002.0,11708.0
1,2024,RLT_10,10001,FIN,8002,50001,PROD,8002.0,12023.0
2,2024,RLT_10,50000,PROD,8002,80070,PROD,8007.0,11303.0
3,2024,RLT_10,50000,PROD,8002,90000,ADD,NaN,598.0
4,2024,RLT_10,50000,PROD,8002,90001,ADD,NaN,242.0


In [51]:
# STEP 5: Build the BoM hierarchy
# Follow PROD components from each FIN material, then add annual quantities

def create_hierarchy(annual_components, annual_production):

    # Store the final rows
    results = []

    # Find all FIN materials
    fins = annual_components[
        annual_components["produced_material_release_type"] == "FIN"
    ]

    # Start with each FIN material
    for _, fin in fins.iterrows():

        queue = [fin["component_material"]]
        visited = set()

        while queue:

            material = queue.pop(0)

            if material in visited:
                continue

            visited.add(material)

            # Find components of the current material
            children = annual_components[
                (annual_components["produced_material"] == material) &
                (annual_components["year"] == fin["year"]) &
                (annual_components["plant_id"] == fin["plant_id"])
            ]

            for _, child in children.iterrows():

                # Save the material relationship
                results.append({
                    "plant": fin["plant_id"],
                    "year": fin["year"],
                    "fin_material_id": fin["produced_material"],
                    "fin_material_release_type": fin["produced_material_release_type"],
                    "fin_material_production_type": fin["produced_material_production_type"],
                    "prod_material_id": child["produced_material"],
                    "prod_material_release_type": child["produced_material_release_type"],
                    "prod_material_production_type": child["produced_material_production_type"],
                    "component_id": child["component_material"],
                    "component_material_release_type": child["component_material_release_type"],
                    "component_material_production_type": child["component_material_production_type"],
                    "component_consumption_quantity": child["component_material_quantity"]
                })

                # If the component is PROD, explore it next
                if child["component_material_release_type"] == "PROD":
                    queue.append(child["component_material"])

    result = pd.DataFrame(results)

    # Add FIN annual production quantities
    fin_quantities = annual_production[
        ["year", "plant_id", "produced_material", "produced_material_quantity"]
    ].rename(columns={
        "plant_id": "plant",
        "produced_material": "fin_material_id",
        "produced_material_quantity": "fin_production_quantity"
    })

    # Add PROD annual production quantities
    prod_quantities = annual_production[
        ["year", "plant_id", "produced_material", "produced_material_quantity"]
    ].rename(columns={
        "plant_id": "plant",
        "produced_material": "prod_material_id",
        "produced_material_quantity": "prod_material_production_quantity"
    })

    result = result.merge(
        fin_quantities,
        on=["year", "plant", "fin_material_id"],
        how="left"
    )

    result = result.merge(
        prod_quantities,
        on=["year", "plant", "prod_material_id"],
        how="left"
    )

    return result

In [52]:
# STEP 6: Run the hierarchy function
# Check the number of rows and columns and preview the result

result = create_hierarchy(
    annual_components,
    annual_production
)

print(result.shape)

display(result.head(10))

(100, 14)


,plant,year,fin_material_id,fin_material_release_type,fin_material_production_type,prod_material_id,prod_material_release_type,prod_material_production_type,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,fin_production_quantity,prod_material_production_quantity
0,RLT_10,2024,10000,FIN,8002,50000,PROD,8002,80070,PROD,8007.0,11303.0,11708.0,9538.0
1,RLT_10,2024,10000,FIN,8002,50000,PROD,8002,90000,ADD,NaN,598.0,11708.0,9538.0
2,RLT_10,2024,10000,FIN,8002,50000,PROD,8002,90001,ADD,NaN,242.0,11708.0,9538.0
3,RLT_10,2024,10000,FIN,8002,80070,PROD,8007,80010,PROD,8001.0,41769.0,11708.0,11028.0
4,RLT_10,2024,10000,FIN,8002,80070,PROD,8007,90002,ADD,NaN,355.0,11708.0,11028.0
5,RLT_10,2024,10000,FIN,8002,80070,PROD,8007,90003,ADD,NaN,121.0,11708.0,11028.0
6,RLT_10,2024,10000,FIN,8002,80010,PROD,8001,80000,PROD,8000.0,23360.0,11708.0,21013.0
7,RLT_10,2024,10000,FIN,8002,80010,PROD,8001,90004,ADD,NaN,1242.0,11708.0,21013.0
8,RLT_10,2024,10000,FIN,8002,80000,PROD,8000,70000,RM,NaN,30498.0,11708.0,23478.0
9,RLT_10,2024,10000,FIN,8002,80000,PROD,8000,90005,ADD,NaN,829.0,11708.0,23478.0


In [53]:
# STEP 7: Validate the result
# Check totals, duplicates and the hierarchy of FIN material 10000

print("Rows:", len(result))
print("Columns:", len(result.columns))
print("Years:", result["year"].unique())
print("Duplicates:", result.duplicated().sum())

display(result[result["fin_material_id"] == 10000])

Rows: 100
Columns: 14
Years: [2024]
Duplicates: 0


,plant,year,fin_material_id,fin_material_release_type,fin_material_production_type,prod_material_id,prod_material_release_type,prod_material_production_type,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,fin_production_quantity,prod_material_production_quantity
0,RLT_10,2024,10000,FIN,8002,50000,PROD,8002,80070,PROD,8007.0,11303.0,11708.0,9538.0
1,RLT_10,2024,10000,FIN,8002,50000,PROD,8002,90000,ADD,NaN,598.0,11708.0,9538.0
2,RLT_10,2024,10000,FIN,8002,50000,PROD,8002,90001,ADD,NaN,242.0,11708.0,9538.0
3,RLT_10,2024,10000,FIN,8002,80070,PROD,8007,80010,PROD,8001.0,41769.0,11708.0,11028.0
4,RLT_10,2024,10000,FIN,8002,80070,PROD,8007,90002,ADD,NaN,355.0,11708.0,11028.0
5,RLT_10,2024,10000,FIN,8002,80070,PROD,8007,90003,ADD,NaN,121.0,11708.0,11028.0
6,RLT_10,2024,10000,FIN,8002,80010,PROD,8001,80000,PROD,8000.0,23360.0,11708.0,21013.0
7,RLT_10,2024,10000,FIN,8002,80010,PROD,8001,90004,ADD,NaN,1242.0,11708.0,21013.0
8,RLT_10,2024,10000,FIN,8002,80000,PROD,8000,70000,RM,NaN,30498.0,11708.0,23478.0
9,RLT_10,2024,10000,FIN,8002,80000,PROD,8000,90005,ADD,NaN,829.0,11708.0,23478.0


In [54]:
# Export the initial result to Excel

result.to_excel("../output/bom_result.xlsx", index=False)

print("BoM hierarchy saved successfully!")

BoM hierarchy saved successfully!


In [55]:
# Additional validation: column names, years and missing values

print(result.shape)
print(result.columns.tolist())
print(result["year"].unique())
print(result.isna().sum())

(100, 14)
['plant', 'year', 'fin_material_id', 'fin_material_release_type', 'fin_material_production_type', 'prod_material_id', 'prod_material_release_type', 'prod_material_production_type', 'component_id', 'component_material_release_type', 'component_material_production_type', 'component_consumption_quantity', 'fin_production_quantity', 'prod_material_production_quantity']
[2024]
plant                                  0
year                                   0
fin_material_id                        0
fin_material_release_type              0
fin_material_production_type           0
prod_material_id                       0
prod_material_release_type             0
prod_material_production_type          0
component_id                           0
component_material_release_type        0
component_material_production_type    70
component_consumption_quantity         0
fin_production_quantity                0
prod_material_production_quantity      0
dtype: int64


In [56]:
# Arrange the 14 required columns and save the final Excel file

columns = [
    "plant",
    "fin_material_id",
    "fin_material_release_type",
    "fin_material_production_type",
    "fin_production_quantity",
    "prod_material_id",
    "prod_material_release_type",
    "prod_material_production_type",
    "prod_material_production_quantity",
    "component_id",
    "component_material_release_type",
    "component_material_production_type",
    "component_consumption_quantity",
    "year"
]

result = result[columns]

result.to_excel("../output/bom_result.xlsx", index=False)

display(result.head())

,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,year
0,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007.0,11303.0,2024
1,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,NaN,598.0,2024
2,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,NaN,242.0,2024
3,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001.0,41769.0,2024
4,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90002,ADD,NaN,355.0,2024


In [57]:
# Check duplicates and the number of unique FIN and PROD materials

print("Duplicate rows:", result.duplicated().sum())

print("FIN materials:", result["fin_material_id"].nunique())

print("PROD materials:", result["prod_material_id"].nunique())

Duplicate rows: 0
FIN materials: 10
PROD materials: 40
